In [4]:
# 1. Импорты для разговорного RAG (LangChain 1.x + LCEL)
import os
import pickle
from dotenv import load_dotenv
from getpass import getpass

# LangChain Community компоненты
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.llms import HuggingFaceEndpoint

# LangChain Core компоненты (новый стандарт)
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory

# Для работы с текстом
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Загружаем переменные окружения
load_dotenv()

import langchain
print(f"✅ LangChain version: {langchain.__version__}")
print("✅ Библиотеки импортированы (современный LCEL подход)")

✅ LangChain version: 1.2.10
✅ Библиотеки импортированы (современный LCEL подход)


In [5]:
# 2. Загружаем векторную базу (функция retriever)
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

# Создаём функцию эмбеддинга (ту же, что использовали при индексации)
embedding = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",
    model_kwargs={'device': 'cpu'}
)

# Загружаем базу (путь относительно корня проекта)
vectordb = Chroma(
    persist_directory="../chroma_db",
    embedding_function=embedding
)

# Создаём retriever — он будет искать 4 наиболее релевантных чанка
retriever = vectordb.as_retriever(search_kwargs={"k": 4})
print(f"✅ База загружена, векторов: {vectordb._collection.count()}")

C:\Users\Китя\AppData\Local\Temp\ipykernel_3228\1307873659.py:6: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding = HuggingFaceEmbeddings(
Loading weights: 100%|██████████| 103/103 [00:01<00:00, 86.94it/s, Materializing param=pooler.dense.weight]                              
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
C:\Users\Китя\AppData\Local\Temp\ipykernel_3228\1307873659.py:12: LangChainDeprecati

✅ База загружена, векторов: 486


In [16]:
# 3. Создаём базовую LCEL-цепочку (с ChatHuggingFace из langchain_huggingface)
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace  # Импортируем всё из одного места!
from langchain_core.output_parsers import StrOutputParser
from getpass import getpass
import os

# --- HuggingFace токен ---
hf_token = os.getenv("HUGGINGFACEHUB_API_TOKEN")
if not hf_token:
    hf_token = getpass("Введите ваш HuggingFace токен: ")
    os.environ["HUGGINGFACEHUB_API_TOKEN"] = hf_token
# -------------------------

# Базовая модель из langchain_huggingface
base_llm = HuggingFaceEndpoint(
    repo_id="HuggingFaceH4/zephyr-7b-beta",
    huggingfacehub_api_token=hf_token,
    temperature=0.3,
    max_new_tokens=512,
    task="text-generation"
)

# Оборачиваем в чат-модель
chat_model = ChatHuggingFace(llm=base_llm)

# Промпт с историей
prompt = ChatPromptTemplate.from_messages([
    ("system", "Ты полезный ассистент. Отвечай на вопросы, используя предоставленный контекст."),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", "{input}")
])

# Собираем LCEL-цепочку
chain = prompt | chat_model | StrOutputParser()
print("✅ Базовая цепочка создана")
print(f"Модель: HuggingFaceH4/zephyr-7b-beta")

✅ Базовая цепочка создана
Модель: HuggingFaceH4/zephyr-7b-beta


In [17]:
# 4. Настройка хранилища для памяти (in-memory словарь)
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory

# Словарь для хранения истории разных сессий
store = {}

def get_session_history(session_id: str) -> BaseChatMessageHistory:
    """Возвращает историю для session_id. Если нет — создаёт новую."""
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

print("✅ Функция get_session_history готова")

✅ Функция get_session_history готова


In [32]:
# 5. ФИНАЛЬНАЯ ВЕРСИЯ: Используем Hugging Face с явным указанием провайдера
from huggingface_hub import InferenceClient
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
import os
from getpass import getpass

# --- HuggingFace токен ---
hf_token = os.getenv("HUGGINGFACEHUB_API_TOKEN")
if not hf_token:
    hf_token = getpass("Введите ваш HuggingFace токен: ")
    os.environ["HUGGINGFACEHUB_API_TOKEN"] = hf_token
# -------------------------

# Используем модель с явным указанием провайдера, который гарантированно работает
client = InferenceClient(
    model="meta-llama/Llama-3.2-3B-Instruct",
    token=hf_token,
    provider="together"  # ЯВНО УКАЗЫВАЕМ ПРОВАЙДЕРА!
)

# Функция для вызова модели с историей
def call_with_history(messages):
    """Отправляет сообщения в модель и получает ответ"""
    try:
        response = client.chat.completions.create(
            messages=messages,
            max_tokens=512,
            temperature=0.3
        )
        return response.choices[0].message.content
    except Exception as e:
        return f"Ошибка: {str(e)}"

# Словарь для хранения истории разных сессий
store = {}

def get_session_history(session_id: str) -> BaseChatMessageHistory:
    """Возвращает историю для session_id"""
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

print("✅ Inference Client готов к работе")
print("✅ Модель: meta-llama/Llama-3.2-3B-Instruct")
print("✅ Провайдер: together (гарантированно работает)")

✅ Inference Client готов к работе
✅ Модель: meta-llama/Llama-3.2-3B-Instruct
✅ Провайдер: together (гарантированно работает)


In [34]:
# 6. Тестируем диалог с новой моделью
session_id = "test-session-1"

# ВАЖНО: сначала инициализируем историю через функцию
history = get_session_history(session_id)

# Вопрос 1
messages_1 = [
    {"role": "user", "content": "Что такое in-context learning?"}
]
print("⏳ Отправляю запрос к модели...")
response1 = call_with_history(messages_1)
print(f"🤖: {response1}\n")

# Сохраняем историю (теперь store[session_id] точно существует)
store[session_id].add_user_message("Что такое in-context learning?")
store[session_id].add_ai_message(response1)

# Вопрос 2 (с контекстом)
messages_2 = [
    {"role": "user", "content": "А кто основные авторы этого направления?"}
]
print("⏳ Отправляю второй запрос...")
response2 = call_with_history(messages_2)
print(f"🤖: {response2}\n")

# Сохраняем историю
store[session_id].add_user_message("А кто основные авторы этого направления?")
store[session_id].add_ai_message(response2)

print(f"📝 История сессии '{session_id}': {len(store[session_id].messages)} сообщений")

⏳ Отправляю запрос к модели...
🤖: Ин-контекстное обучение (in-context learning) - это метод обучения, в котором обучение происходит в контексте, в котором будет применено знания или навыки. Это означает, что обучение происходит в реальных условиях, а не в теоретическом или контролируемом окружении.

В ин-контекстном обучении учащиеся или учащиеся обучаются на практике, решая реальные проблемы или completing задачи, связанные с темой, которую они изучают. Этот подход позволяет учащимся применять свои знания и навыки в реальных ситуациях, что приводит к более глубокому пониманию предмета и более эффективному применению знаний.

Ин-контекстное обучение имеет कई преимущества, такие как:

* Более эффективное применение знаний: ин-контекстное обучение позволяет учащимся применять свои знания в реальных ситуациях, что приводит к более эффективному применению знаний.
* Более глубокое понимание предмета: ин-контекстное обучение позволяет учащимся получить более глубокое понимание предмета, поск